In [1]:
import os
import re
import logging
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# --- Configuration ---
# ** SET YOUR FILE PATHS HERE **
METADATA_CSV_PATH = '/data/users4/nshaik3/Datasets/MIMIC-CXR/physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-metadata.csv'
SPLIT_CSV_PATH = '/data/users4/nshaik3/Datasets/MIMIC-CXR/physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-split.csv'
# Base directory where the report/image 'files' folder resides
BASE_MIMIC_DIR = '/data/users4/nshaik3/Datasets/MIMIC-CXR/physionet.org/files/mimic-cxr-jpg/2.1.0/'
OUTPUT_CSV_PATH = '/data/users4/nshaik3/Datasets/MIMIC-CXR/physionet.org/files/mimic-cxr-jpg/2.1.0/longitudinal_mimic-cxr-v1.0.csv'

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logging.getLogger().setLevel(logging.DEBUG)
tqdm.pandas() # Enable progress_apply for pandas


# --- Section Parsing Functions ---
def section_text(text):
    """Splits text into sections."""
    p_section = re.compile(r'\n ([A-Z ()/,-]+):\s', re.DOTALL)
    sections, section_names, section_idx = [], [], []
    idx = 0
    s = p_section.search(text, idx)

    if s:
        s_content = str(text[0:s.start(1)]).strip()
        sections.append(re.sub(r"\s+", " ", s_content))
        section_names.append('preamble')
        section_idx.append(0)
        while s:
            current_section = s.group(1).lower()
            idx_start = s.end()
            idx_skip = text[idx_start:].find('\n')
            if idx_skip == -1: idx_skip = 0
            s = p_section.search(text, idx_start + idx_skip)
            idx_end = s.start() if s else len(text)
            s_content = str(text[idx_start:idx_end]).strip()
            sections.append(re.sub(r"\s+", " ", s_content))
            section_names.append(current_section)
            section_idx.append(idx_start)
    else:
        sections.append(re.sub(r"\s+", " ", str(text).strip()))
        section_names.append('full report')
        section_idx.append(0)

    section_names = normalize_section_names(section_names)

    # Remove empty findings/impression sections
    for i in reversed(range(len(section_names))):
        if section_names[i] in ('impression', 'findings'):
            if not sections[i].strip():
                sections.pop(i); section_names.pop(i); section_idx.pop(i)

    if ('impression' not in section_names) and ('findings' not in section_names):
         if len(sections) > 0 and '\n \n' in sections[-1]:
             split_text = sections[-1].split('\n \n')
             if len(split_text) > 1:
                 sections.append('\n \n'.join(split_text[1:]))
                 sections[-2] = split_text[0]
                 section_names.append('last_paragraph')
                 if section_idx: # Check if section_idx is not empty
                    section_idx.append(section_idx[-1] + len(sections[-2]))
                 else: # Handle case where section_idx might be empty initially
                    section_idx.append(len(sections[-2]))

    return sections, section_names, section_idx

def normalize_section_names(section_names):
    """Normalizes common section header variations."""
    section_names = [s.lower().strip() for s in section_names]
    frequent_sections = {
        "preamble": "preamble", "impression": "impression", "comparison": "comparison",
        "indication": "indication", "findings": "findings", "examination": "examination",
        "technique": "technique", "history": "history", "comparisons": "comparison",
        "clinical history": "history", "reason for examination": "indication",
        "notification": "notification", "reason for exam": "indication",
        "clinical information": "history", "exam": "examination",
        "clinical indication": "indication", "conclusion": "impression",
        "chest, two views": "findings", "recommendation(s)": "recommendations",
        "type of examination": "examination", "reference exam": "comparison",
        "patient history": "history", "addendum": "addendum", "comparison exam": "comparison",
        "date": "date", "comment": "comment", "findings and impression": "impression",
        "wet read": "wet read", "comparison film": "comparison", "recommendations": "recommendations",
        "findings/impression": "impression", "pfi": "history", 'recommendation': 'recommendations',
        'wetread': 'wet read', # Typos/Variations below
        'ndication': 'impression', 'impresson': 'impression', 'imprression': 'impression',
        'imoression': 'impression', 'impressoin': 'impression', 'imprssion': 'impression',
        'impresion': 'impression', 'imperssion': 'impression', 'mpression': 'impression',
        'impession': 'impression', 'findings/ impression': 'impression', 'finding': 'findings',
        'findins': 'findings', 'findindgs': 'findings', 'findgings': 'findings',
        'findngs': 'findings', 'findnings': 'findings', 'finidngs': 'findings',
        'idication': 'indication', 'reference findings': 'findings', 'comparision': 'comparison',
        'comparsion': 'comparison', 'comparrison': 'comparison', 'comparisions': 'comparison'
    }
    p_findings = re.compile(
        r'(chest|portable|pa and lateral|lateral and pa|ap and lateral|lateral and ap|frontal and|two views|'
        r'frontal view|pa view|ap view|one view|lateral view|bone window|frontal upright|'
        r'frontal semi-upright|ribs|pa and lat)'
    )
    main_sections = ['impression', 'findings', 'history', 'comparison', 'addendum']

    for i, s in enumerate(section_names):
        if s in frequent_sections: section_names[i] = frequent_sections[s]; continue
        main_flag = False
        for m in main_sections:
            if m in s: section_names[i] = m; main_flag = True; break
        if main_flag: continue
        if p_findings.search(s): section_names[i] = 'findings'
    return section_names

# --- Path Construction Helper ---
def construct_relative_paths(row):
    """Constructs relative image and report paths (e.g., files/p10/...)."""
    try:
        subject_id_str = str(int(row['subject_id']))
        study_id_str = str(int(row['study_id']))
        dicom_id_str = str(row['dicom_id'])
        p_group = "p" + subject_id_str[:2]
        subject_dir = "p" + subject_id_str
        study_dir = "s" + study_id_str
        # Construct relative paths starting from 'files'
        report_rel_path = os.path.join('files', p_group, subject_dir, f"{study_dir}.txt")
        image_rel_path = os.path.join('files', p_group, subject_dir, study_dir, f"{dicom_id_str}.jpg")
        # Standardize path separators for consistency (optional but good)
        report_rel_path = report_rel_path.replace('\\', '/')
        image_rel_path = image_rel_path.replace('\\', '/')
        return pd.Series([image_rel_path, report_rel_path])
    except Exception as e:
        logging.warning(f"Error constructing relative path for row {row.name}: {e}")
        return pd.Series([None, None])

# --- Anchor/Auxiliary Helper ---
def find_anchor_auxiliary(image_paths, view_positions):
    """Identifies anchor (PA/AP preferred) and auxiliary images/views."""
    # (Same implementation as before)
    if not image_paths or not view_positions or len(image_paths) != len(view_positions):
        return None, None, [], []
    anchor_idx, pa_idx, ap_idx = -1, -1, -1
    for i, view in enumerate(view_positions):
        view_upper = str(view).upper()
        if view_upper == 'PA' and pa_idx == -1: pa_idx = i; break
        elif view_upper == 'AP' and ap_idx == -1: ap_idx = i
    if pa_idx != -1: anchor_idx = pa_idx
    elif ap_idx != -1: anchor_idx = ap_idx
    elif image_paths: anchor_idx = 0 # Fallback only if paths exist
    else: return None, None, [], [] # No images to select from

    anchor_image = image_paths[anchor_idx]
    anchor_view = view_positions[anchor_idx]
    aux_images = [img for i, img in enumerate(image_paths) if i != anchor_idx]
    aux_views = [view for i, view in enumerate(view_positions) if i != anchor_idx]
    return anchor_image, anchor_view, aux_images, aux_views


# --- Report Section Processing (Accepts Base Dir & Relative Path) ---
def process_report(base_dir, relative_report_path):
    """Reads report (using absolute path), extracts sections, and cleans text."""
    extracted_sections = {'indication': "", 'findings': "", 'impression': ""}
    if not isinstance(relative_report_path, str) or not relative_report_path:
        logging.debug("Invalid or empty relative report path provided.")
        return pd.Series(["", "", ""]) # Return empty sections

    # Construct absolute path for reading
    absolute_report_path = os.path.join(base_dir, relative_report_path)

    try:
        # Attempt to read with UTF-8 first, fallback to latin-1 if needed
        try:
            with open(absolute_report_path, 'r', encoding='utf-8') as f:
                report_text = f.read()
        except UnicodeDecodeError:
            logging.debug(f"UTF-8 failed for {absolute_report_path}, trying latin-1.")
            with open(absolute_report_path, 'r', encoding='latin-1') as f:
                report_text = f.read()

        sections, section_names, _ = section_text(report_text)

        for i, name in enumerate(section_names):
            if name in extracted_sections:
                text = sections[i]
                if extracted_sections[name]:
                     extracted_sections[name] += " . " + text
                else:
                     extracted_sections[name] = text

    except FileNotFoundError:
        logging.warning(f"Report file not found at absolute path: {absolute_report_path} (derived from relative: {relative_report_path})")
    except Exception as e:
        logging.error(f"Error processing report {absolute_report_path}: {e}")

    return pd.Series([
        extracted_sections['indication'],
        extracted_sections['findings'],
        extracted_sections['impression']
    ])

# --- Main Processing Function ---
def prepare_longitudinal_mimiccxr_dataset(metadata_path, split_path, base_mimic_dir, output_path):
    """
    Generates detailed CSV with anchor/aux views, RELATIVE paths,
    AND processed report sections using revised cleaning.
    """
    logging.info("Starting detailed longitudinal data preparation (relative paths, revised cleaning)...")

    # --- 1. Load Data ---
    logging.info(f"Loading metadata: {metadata_path}")
    try:
        meta_cols = ['dicom_id', 'subject_id', 'study_id', 'StudyDate', 'ViewPosition']
        meta_dtypes = {'subject_id': int, 'study_id': int, 'dicom_id': str, 'ViewPosition': str}
        metadata_df = pd.read_csv(metadata_path, usecols=meta_cols, dtype=meta_dtypes, low_memory=False)
        metadata_df.rename(columns={'ViewPosition': 'view_position'}, inplace=True)
    except Exception as e: logging.error(f"Metadata load failed: {e}"); return False
    logging.info(f"Loading splits: {split_path}")
    try:
        split_cols = ['dicom_id', 'subject_id', 'study_id', 'split']
        split_dtypes = {'subject_id': int, 'study_id': int, 'dicom_id': str, 'split': str}
        split_df = pd.read_csv(split_path, usecols=split_cols, dtype=split_dtypes)
    except Exception as e: logging.error(f"Split load failed: {e}"); return False

    # --- 2. Merge ---
    logging.info("Merging metadata and split info...")
    merge_cols = ['dicom_id', 'subject_id', 'study_id']
    merged_df = pd.merge(metadata_df, split_df, on=merge_cols, how='inner')
    logging.info(f"Merged images: {len(merged_df)}")
    if merged_df.empty: logging.error("Merge failed."); return False


    # --- 3. Clean ---
    logging.info("Cleaning merged data...")
    merged_df.dropna(subset=['subject_id', 'study_id', 'StudyDate', 'dicom_id'], inplace=True)
    merged_df['view_position'].fillna('UNKNOWN', inplace=True)
    try:
        merged_df['StudyDate'] = pd.to_datetime(merged_df['StudyDate'].astype(str), format='%Y%m%d', errors='coerce')
        merged_df.dropna(subset=['StudyDate'], inplace=True)
    except Exception as e: logging.error(f"Date conversion error: {e}"); return False
    logging.info(f"Cleaned images: {len(merged_df)}")
    if merged_df.empty: logging.error("No data after cleaning."); return False

    # --- 4. Construct RELATIVE Paths ---
    logging.info("Constructing relative image and report file paths...")
    path_cols = ['image_path', 'report_path'] # These will store relative paths
    merged_df[path_cols] = merged_df.progress_apply(construct_relative_paths, axis=1)
    merged_df.dropna(subset=path_cols, inplace=True)
    logging.info(f"Constructed relative paths for {len(merged_df)} images.")
    if merged_df.empty: logging.error("No data after path construction."); return False

    # --- 5. Sort (Image Level) ---
    logging.info("Sorting images chronologically...")
    merged_df.sort_values(by=['subject_id', 'StudyDate', 'study_id', 'dicom_id'], inplace=True)


    # --- 6. Aggregate to Study Level & Find Anchor/Aux ---
    logging.info("Aggregating studies & identifying anchor/aux views...")
    # (Aggregation logic is the same, but now aggregates RELATIVE paths)
    def aggregate_and_process_study(group):
        study_date = group['StudyDate'].iloc[0]
        report_path = group['report_path'].iloc[0] # This is now the relative path
        unique_splits = group['split'].unique()
        split = unique_splits[0]
        image_paths = list(group['image_path']) # List of relative image paths
        view_positions = list(group['view_position'])
        anchor_image, anchor_view, aux_images, aux_views = find_anchor_auxiliary(image_paths, view_positions)
        return pd.Series({
            'study_date': study_date, 'report_path': report_path, 'split': split,
            'anchor_image': anchor_image, 'anchor_view': anchor_view,
            'aux_images': aux_images, 'aux_views': aux_views
        })

    study_agg_df = merged_df.groupby(['subject_id', 'study_id'], observed=True, sort=False).progress_apply(aggregate_and_process_study)
    study_agg_df = study_agg_df.reset_index()
    logging.info(f"Aggregated {len(study_agg_df)} unique studies.")


    # --- 7. Sort Studies ---
    logging.info("Sorting studies chronologically...")
    study_agg_df.sort_values(by=['subject_id', 'study_date', 'study_id'], inplace=True)


    # --- 8. Identify Longitudinal Patients ---
    logging.info("Identifying longitudinal studies...")
    study_counts = study_agg_df.groupby('subject_id')['study_id'].transform('nunique')
    longitudinal_studies = study_agg_df[study_counts >= 2].copy()
    num_long_patients = longitudinal_studies['subject_id'].nunique()
    logging.info(f"Found {num_long_patients} patients with >= 2 studies.")
    if longitudinal_studies.empty: logging.warning("No longitudinal patients."); return True


    # --- 9. Create Previous Visit Columns ---
    logging.info("Creating previous visit columns...")
    groupby_subject = longitudinal_studies.groupby('subject_id')
    cols_to_shift = [
        'study_id', 'study_date', 'report_path', 'split',
        'anchor_image', 'anchor_view', 'aux_images', 'aux_views'
    ]
    for col in cols_to_shift:
        longitudinal_studies[f'prior_{col}'] = groupby_subject[col].shift(1)


    # --- 10. Filter out first visits ---
    logging.info("Filtering out first visits...")
    final_pairs = longitudinal_studies.dropna(subset=['prior_study_id']).copy()
    num_samples = len(final_pairs)
    logging.info(f"Generated {num_samples} longitudinal pairs.")


    # --- 11. Process Reports using Base Dir and Relative Paths ---
    logging.info("Processing current study reports...")
    report_cols = ['current_indication', 'current_findings', 'current_impression']
    final_pairs[report_cols] = final_pairs['report_path'].progress_apply(
        lambda rel_path: process_report(base_mimic_dir, rel_path)
    )

    logging.info("Processing previous study reports...")
    prev_report_cols = ['prior_indication', 'prior_findings', 'prior_impression']
    final_pairs[prev_report_cols] = final_pairs['prior_report_path'].progress_apply(
        lambda rel_path: process_report(base_mimic_dir, rel_path)
    )

    # --- 12. Prepare Final Output (with Relative Paths) ---
    logging.info("Formatting final data for CSV...")
    final_pairs.rename(columns={
        'study_id': 'current_study_id', 'study_date': 'current_study_date',
        'report_path': 'current_report_path', # Stays relative path
        'split': 'current_split',
        'anchor_image': 'current_anchor_image', # Stays relative path
        'anchor_view': 'current_anchor_view',
        'aux_images': 'current_aux_images', # List of relative paths
        'aux_views': 'current_aux_views'
    }, inplace=True)

    # Select and order columns
    output_columns = [
        'subject_id',
        'current_study_id', 'current_study_date', 'current_report_path', 'current_split',
        'current_anchor_image', 'current_anchor_view', 'current_aux_images', 'current_aux_views',
        'current_indication', 'current_findings', 'current_impression',
        'prior_study_id', 'prior_study_date', 'prior_report_path', 'prior_split',
        'prior_anchor_image', 'prior_anchor_view', 'prior_aux_images', 'prior_aux_views',
        'prior_indication', 'prior_findings', 'prior_impression'
    ]
    final_output_df = final_pairs[[col for col in output_columns if col in final_pairs.columns]]

    # Convert types and format lists
    final_output_df = final_output_df.astype({
        'current_study_id': int, 'prior_study_id': int,
        'current_anchor_view': str, 'prior_anchor_view': str,
        'current_indication': str, 'current_findings': str, 'current_impression': str,
        'prior_indication': str, 'prior_findings': str, 'prior_impression': str,
        # Ensure path columns are strings (they should be already)
        'current_report_path': str, 'prior_report_path': str,
        'current_anchor_image': str, 'prior_anchor_image': str,
    })
    final_output_df['current_study_date'] = final_output_df['current_study_date'].dt.strftime('%Y-%m-%d')
    final_output_df['prior_study_date'] = final_output_df['prior_study_date'].dt.strftime('%Y-%m-%d')
    list_delimiter = '|'
    for col in ['current_aux_images', 'current_aux_views', 'prior_aux_images', 'prior_aux_views']:
         if col in final_output_df.columns:
             final_output_df[col] = final_output_df[col].apply(
                 lambda x: list_delimiter.join(map(str, x)) if isinstance(x, list) and x else ''
             ).astype(str) # Ensure final column is string

    # --- 13. Save ---
    logging.info(f"Saving final detailed report data with relative paths to: {output_path}")
    try:
        final_output_df.to_csv(output_path, index=False)
        logging.info("CSV file saved successfully.")
        return True
    except Exception as e:
        logging.error(f"Error saving data to CSV: {e}")
        return False

# --- Execute ---
if __name__ == '__main__':
    # Validate paths
    if not os.path.isfile(METADATA_CSV_PATH): logging.error(f"Metadata file missing: {METADATA_CSV_PATH}"); exit(1)
    if not os.path.isfile(SPLIT_CSV_PATH): logging.error(f"Split file missing: {SPLIT_CSV_PATH}"); exit(1)
    if not os.path.isdir(BASE_MIMIC_DIR): logging.error(f"Base MIMIC directory not found: {BASE_MIMIC_DIR}"); exit(1)
    # Check if the 'files' subdirectory exists under base dir
    files_dir_path = os.path.join(BASE_MIMIC_DIR, 'files')
    if not os.path.isdir(files_dir_path): logging.error(f"'files' directory not found under BASE_MIMIC_DIR: {files_dir_path}"); exit(1)

    success = prepare_longitudinal_mimiccxr_dataset(
        METADATA_CSV_PATH, SPLIT_CSV_PATH, BASE_MIMIC_DIR, OUTPUT_CSV_PATH
    )

    if success:
        print(f"\nSuccessfully created detailed longitudinal CSV with relative paths and revised report cleaning: {OUTPUT_CSV_PATH}")
    else:
        print("\nFailed to create detailed longitudinal CSV.")
        exit(1)

/home/users/nshaik3/miniconda3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-04-26 20:39:35,117 - INFO - Starting detailed longitudinal data preparation (relative paths, revised cleaning)...
2025-04-26 20:39:35,118 - INFO - Loading metadata: /data/users4/nshaik3/Datasets/MIMIC-CXR/physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-metadata.csv
2025-04-26 20:39:36,139 - INFO - Loading splits: /data/users4/nshaik3/Datasets/MIMIC-CXR/physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-split.csv
2025-04-26 20:39:36,621 - INFO - Merging metadata and split info...
2025-04-26 20:39:37,135 - INFO - Merged images: 377110
2025-04-26 20:39:37,137 - INFO - Cleaning merged data...
2025-04-26 20:39:37,696 - INFO - Cleaned images: 377110
2025-04-26 20:39:37,698 - INFO - Constructing relative image and 


Successfully created detailed longitudinal CSV with relative paths and revised report cleaning: /data/users4/nshaik3/Datasets/MIMIC-CXR/physionet.org/files/mimic-cxr-jpg/2.1.0/longitudinal_mimic-cxr-v1.0.csv


In [ ]:
import os
import re
import logging
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# --- Configuration ---
# ** SET YOUR FILE PATHS HERE **
METADATA_CSV_PATH = '/data/users4/nshaik3/Datasets/MIMIC-CXR/physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-metadata.csv'
SPLIT_CSV_PATH = '/data/users4/nshaik3/Datasets/MIMIC-CXR/physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-split.csv'
# Base directory where the report/image 'files' folder resides
BASE_MIMIC_DIR = '/data/users4/nshaik3/Datasets/MIMIC-CXR/physionet.org/files/mimic-cxr-jpg/2.1.0/'
OUTPUT_CSV_PATH = '/data/users4/nshaik3/Datasets/MIMIC-CXR/physionet.org/files/mimic-cxr-jpg/2.1.0/longitudinal_mimic-cxr-v1.0.csv'

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logging.getLogger().setLevel(logging.DEBUG)
tqdm.pandas() # Enable progress_apply for pandas


# --- Section Parsing Functions ---
def section_text(text):
    """Splits text into sections."""
    p_section = re.compile(r'\n ([A-Z ()/,-]+):\s', re.DOTALL)
    sections, section_names, section_idx = [], [], []
    idx = 0
    s = p_section.search(text, idx)

    if s:
        s_content = str(text[0:s.start(1)]).strip()
        sections.append(re.sub(r"\s+", " ", s_content))
        section_names.append('preamble')
        section_idx.append(0)
        while s:
            current_section = s.group(1).lower()
            idx_start = s.end()
            idx_skip = text[idx_start:].find('\n')
            if idx_skip == -1: idx_skip = 0
            s = p_section.search(text, idx_start + idx_skip)
            idx_end = s.start() if s else len(text)
            s_content = str(text[idx_start:idx_end]).strip()
            sections.append(re.sub(r"\s+", " ", s_content))
            section_names.append(current_section)
            section_idx.append(idx_start)
    else:
        sections.append(re.sub(r"\s+", " ", str(text).strip()))
        section_names.append('full report')
        section_idx.append(0)

    section_names = normalize_section_names(section_names)

    # Remove empty findings/impression sections
    for i in reversed(range(len(section_names))):
        if section_names[i] in ('impression', 'findings'):
            if not sections[i].strip():
                sections.pop(i); section_names.pop(i); section_idx.pop(i)

    if ('impression' not in section_names) and ('findings' not in section_names):
         if len(sections) > 0 and '\n \n' in sections[-1]:
             split_text = sections[-1].split('\n \n')
             if len(split_text) > 1:
                 sections.append('\n \n'.join(split_text[1:]))
                 sections[-2] = split_text[0]
                 section_names.append('last_paragraph')
                 if section_idx: # Check if section_idx is not empty
                    section_idx.append(section_idx[-1] + len(sections[-2]))
                 else: # Handle case where section_idx might be empty initially
                    section_idx.append(len(sections[-2]))

    return sections, section_names, section_idx

def normalize_section_names(section_names):
    """Normalizes common section header variations."""
    section_names = [s.lower().strip() for s in section_names]
    frequent_sections = {
        "preamble": "preamble", "impression": "impression", "comparison": "comparison",
        "indication": "indication", "findings": "findings", "examination": "examination",
        "technique": "technique", "history": "history", "comparisons": "comparison",
        "clinical history": "history", "reason for examination": "indication",
        "notification": "notification", "reason for exam": "indication",
        "clinical information": "history", "exam": "examination",
        "clinical indication": "indication", "conclusion": "impression",
        "chest, two views": "findings", "recommendation(s)": "recommendations",
        "type of examination": "examination", "reference exam": "comparison",
        "patient history": "history", "addendum": "addendum", "comparison exam": "comparison",
        "date": "date", "comment": "comment", "findings and impression": "impression",
        "wet read": "wet read", "comparison film": "comparison", "recommendations": "recommendations",
        "findings/impression": "impression", "pfi": "history", 'recommendation': 'recommendations',
        'wetread': 'wet read', # Typos/Variations below
        'ndication': 'impression', 'impresson': 'impression', 'imprression': 'impression',
        'imoression': 'impression', 'impressoin': 'impression', 'imprssion': 'impression',
        'impresion': 'impression', 'imperssion': 'impression', 'mpression': 'impression',
        'impession': 'impression', 'findings/ impression': 'impression', 'finding': 'findings',
        'findins': 'findings', 'findindgs': 'findings', 'findgings': 'findings',
        'findngs': 'findings', 'findnings': 'findings', 'finidngs': 'findings',
        'idication': 'indication', 'reference findings': 'findings', 'comparision': 'comparison',
        'comparsion': 'comparison', 'comparrison': 'comparison', 'comparisions': 'comparison'
    }
    p_findings = re.compile(
        r'(chest|portable|pa and lateral|lateral and pa|ap and lateral|lateral and ap|frontal and|two views|'
        r'frontal view|pa view|ap view|one view|lateral view|bone window|frontal upright|'
        r'frontal semi-upright|ribs|pa and lat)'
    )
    main_sections = ['impression', 'findings', 'history', 'comparison', 'addendum']

    for i, s in enumerate(section_names):
        if s in frequent_sections: section_names[i] = frequent_sections[s]; continue
        main_flag = False
        for m in main_sections:
            if m in s: section_names[i] = m; main_flag = True; break
        if main_flag: continue
        if p_findings.search(s): section_names[i] = 'findings'
    return section_names

# --- Path Construction Helper ---
def construct_relative_paths(row):
    """Constructs relative image and report paths (e.g., files/p10/...)."""
    try:
        subject_id_str = str(int(row['subject_id']))
        study_id_str = str(int(row['study_id']))
        dicom_id_str = str(row['dicom_id'])
        p_group = "p" + subject_id_str[:2]
        subject_dir = "p" + subject_id_str
        study_dir = "s" + study_id_str
        # Construct relative paths starting from 'files'
        report_rel_path = os.path.join('files', p_group, subject_dir, f"{study_dir}.txt")
        image_rel_path = os.path.join('files', p_group, subject_dir, study_dir, f"{dicom_id_str}.jpg")
        # Standardize path separators for consistency (optional but good)
        report_rel_path = report_rel_path.replace('\\', '/')
        image_rel_path = image_rel_path.replace('\\', '/')
        return pd.Series([image_rel_path, report_rel_path])
    except Exception as e:
        logging.warning(f"Error constructing relative path for row {row.name}: {e}")
        return pd.Series([None, None])

# --- Anchor/Auxiliary Helper ---
def find_anchor_auxiliary(image_paths, view_positions):
    """Identifies anchor (PA/AP preferred) and auxiliary images/views."""
    # (Same implementation as before)
    if not image_paths or not view_positions or len(image_paths) != len(view_positions):
        return None, None, [], []
    anchor_idx, pa_idx, ap_idx = -1, -1, -1
    for i, view in enumerate(view_positions):
        view_upper = str(view).upper()
        if view_upper == 'PA' and pa_idx == -1: pa_idx = i; break
        elif view_upper == 'AP' and ap_idx == -1: ap_idx = i
    if pa_idx != -1: anchor_idx = pa_idx
    elif ap_idx != -1: anchor_idx = ap_idx
    elif image_paths: anchor_idx = 0 # Fallback only if paths exist
    else: return None, None, [], [] # No images to select from

    anchor_image = image_paths[anchor_idx]
    anchor_view = view_positions[anchor_idx]
    aux_images = [img for i, img in enumerate(image_paths) if i != anchor_idx]
    aux_views = [view for i, view in enumerate(view_positions) if i != anchor_idx]
    return anchor_image, anchor_view, aux_images, aux_views


# --- Report Section Processing (Accepts Base Dir & Relative Path) ---
def process_report(base_dir, relative_report_path):
    """Reads report (using absolute path), extracts sections, and cleans text."""
    extracted_sections = {'indication': "", 'findings': "", 'impression': ""}
    if not isinstance(relative_report_path, str) or not relative_report_path:
        logging.debug("Invalid or empty relative report path provided.")
        return pd.Series(["", "", ""]) # Return empty sections

    # Construct absolute path for reading
    absolute_report_path = os.path.join(base_dir, relative_report_path)

    try:
        # Attempt to read with UTF-8 first, fallback to latin-1 if needed
        try:
            with open(absolute_report_path, 'r', encoding='utf-8') as f:
                report_text = f.read()
        except UnicodeDecodeError:
            logging.debug(f"UTF-8 failed for {absolute_report_path}, trying latin-1.")
            with open(absolute_report_path, 'r', encoding='latin-1') as f:
                report_text = f.read()

        sections, section_names, _ = section_text(report_text)

        for i, name in enumerate(section_names):
            if name in extracted_sections:
                text = sections[i]
                if extracted_sections[name]:
                     extracted_sections[name] += " . " + text
                else:
                     extracted_sections[name] = text

    except FileNotFoundError:
        logging.warning(f"Report file not found at absolute path: {absolute_report_path} (derived from relative: {relative_report_path})")
    except Exception as e:
        logging.error(f"Error processing report {absolute_report_path}: {e}")

    return pd.Series([
        extracted_sections['indication'],
        extracted_sections['findings'],
        extracted_sections['impression']
    ])

# --- Main Processing Function ---
def prepare_longitudinal_mimiccxr_dataset(metadata_path, split_path, base_mimic_dir, output_path):
    """
    Generates detailed CSV with anchor/aux views, RELATIVE paths,
    AND processed report sections using revised cleaning.
    """
    logging.info("Starting detailed longitudinal data preparation (relative paths, revised cleaning)...")

    # --- 1. Load Data ---
    logging.info(f"Loading metadata: {metadata_path}")
    try:
        meta_cols = ['dicom_id', 'subject_id', 'study_id', 'StudyDate', 'ViewPosition']
        meta_dtypes = {'subject_id': int, 'study_id': int, 'dicom_id': str, 'ViewPosition': str}
        metadata_df = pd.read_csv(metadata_path, usecols=meta_cols, dtype=meta_dtypes, low_memory=False)
        metadata_df.rename(columns={'ViewPosition': 'view_position'}, inplace=True)
    except Exception as e: logging.error(f"Metadata load failed: {e}"); return False
    logging.info(f"Loading splits: {split_path}")
    try:
        split_cols = ['dicom_id', 'subject_id', 'study_id', 'split']
        split_dtypes = {'subject_id': int, 'study_id': int, 'dicom_id': str, 'split': str}
        split_df = pd.read_csv(split_path, usecols=split_cols, dtype=split_dtypes)
    except Exception as e: logging.error(f"Split load failed: {e}"); return False

    # --- 2. Merge ---
    logging.info("Merging metadata and split info...")
    merge_cols = ['dicom_id', 'subject_id', 'study_id']
    merged_df = pd.merge(metadata_df, split_df, on=merge_cols, how='inner')
    logging.info(f"Merged images: {len(merged_df)}")
    if merged_df.empty: logging.error("Merge failed."); return False


    # --- 3. Clean ---
    logging.info("Cleaning merged data...")
    merged_df.dropna(subset=['subject_id', 'study_id', 'StudyDate', 'dicom_id'], inplace=True)
    merged_df['view_position'].fillna('UNKNOWN', inplace=True)
    try:
        merged_df['StudyDate'] = pd.to_datetime(merged_df['StudyDate'].astype(str), format='%Y%m%d', errors='coerce')
        merged_df.dropna(subset=['StudyDate'], inplace=True)
    except Exception as e: logging.error(f"Date conversion error: {e}"); return False
    logging.info(f"Cleaned images: {len(merged_df)}")
    if merged_df.empty: logging.error("No data after cleaning."); return False

    # --- 4. Construct RELATIVE Paths ---
    logging.info("Constructing relative image and report file paths...")
    path_cols = ['image_path', 'report_path'] # These will store relative paths
    merged_df[path_cols] = merged_df.progress_apply(construct_relative_paths, axis=1)
    merged_df.dropna(subset=path_cols, inplace=True)
    logging.info(f"Constructed relative paths for {len(merged_df)} images.")
    if merged_df.empty: logging.error("No data after path construction."); return False

    # --- 5. Sort (Image Level) ---
    logging.info("Sorting images chronologically...")
    merged_df.sort_values(by=['subject_id', 'StudyDate', 'study_id', 'dicom_id'], inplace=True)


    # --- 6. Aggregate to Study Level & Find Anchor/Aux ---
    logging.info("Aggregating studies & identifying anchor/aux views...")
    # (Aggregation logic is the same, but now aggregates RELATIVE paths)
    def aggregate_and_process_study(group):
        study_date = group['StudyDate'].iloc[0]
        report_path = group['report_path'].iloc[0] # This is now the relative path
        unique_splits = group['split'].unique()
        split = unique_splits[0]
        image_paths = list(group['image_path']) # List of relative image paths
        view_positions = list(group['view_position'])
        anchor_image, anchor_view, aux_images, aux_views = find_anchor_auxiliary(image_paths, view_positions)
        return pd.Series({
            'study_date': study_date, 'report_path': report_path, 'split': split,
            'anchor_image': anchor_image, 'anchor_view': anchor_view,
            'aux_images': aux_images, 'aux_views': aux_views
        })

    study_agg_df = merged_df.groupby(['subject_id', 'study_id'], observed=True, sort=False).progress_apply(aggregate_and_process_study)
    study_agg_df = study_agg_df.reset_index()
    logging.info(f"Aggregated {len(study_agg_df)} unique studies.")


    # --- 7. Sort Studies ---
    logging.info("Sorting studies chronologically...")
    study_agg_df.sort_values(by=['subject_id', 'study_date', 'study_id'], inplace=True)


    # --- 8. Identify Longitudinal Patients ---
    logging.info("Identifying longitudinal studies...")
    study_counts = study_agg_df.groupby('subject_id')['study_id'].transform('nunique')
    longitudinal_studies = study_agg_df[study_counts >= 2].copy()
    num_long_patients = longitudinal_studies['subject_id'].nunique()
    logging.info(f"Found {num_long_patients} patients with >= 2 studies.")
    if longitudinal_studies.empty: logging.warning("No longitudinal patients."); return True


    # --- 9. Create Previous Visit Columns ---
    logging.info("Creating previous visit columns...")
    groupby_subject = longitudinal_studies.groupby('subject_id')
    cols_to_shift = [
        'study_id', 'study_date', 'report_path', 'split',
        'anchor_image', 'anchor_view', 'aux_images', 'aux_views'
    ]
    for col in cols_to_shift:
        longitudinal_studies[f'prior_{col}'] = groupby_subject[col].shift(1)


    # --- 10. Filter out first visits ---
    logging.info("Filtering out first visits...")
    final_pairs = longitudinal_studies.dropna(subset=['prior_study_id']).copy()
    num_samples = len(final_pairs)
    logging.info(f"Generated {num_samples} longitudinal pairs.")


    # --- 11. Process Reports using Base Dir and Relative Paths ---
    logging.info("Processing current study reports...")
    report_cols = ['current_indication', 'current_findings', 'current_impression']
    final_pairs[report_cols] = final_pairs['report_path'].progress_apply(
        lambda rel_path: process_report(base_mimic_dir, rel_path)
    )

    logging.info("Processing previous study reports...")
    prev_report_cols = ['prior_indication', 'prior_findings', 'prior_impression']
    final_pairs[prev_report_cols] = final_pairs['prior_report_path'].progress_apply(
        lambda rel_path: process_report(base_mimic_dir, rel_path)
    )

    # --- 12. Prepare Final Output (with Relative Paths) ---
    logging.info("Formatting final data for CSV...")
    final_pairs.rename(columns={
        'study_id': 'current_study_id', 'study_date': 'current_study_date',
        'report_path': 'current_report_path', # Stays relative path
        'split': 'current_split',
        'anchor_image': 'current_anchor_image', # Stays relative path
        'anchor_view': 'current_anchor_view',
        'aux_images': 'current_aux_images', # List of relative paths
        'aux_views': 'current_aux_views'
    }, inplace=True)

    # Select and order columns
    output_columns = [
        'subject_id',
        'current_study_id', 'current_study_date', 'current_report_path', 'current_split',
        'current_anchor_image', 'current_anchor_view', 'current_aux_images', 'current_aux_views',
        'current_indication', 'current_findings', 'current_impression',
        'prior_study_id', 'prior_study_date', 'prior_report_path', 'prior_split',
        'prior_anchor_image', 'prior_anchor_view', 'prior_aux_images', 'prior_aux_views',
        'prior_indication', 'prior_findings', 'prior_impression'
    ]
    final_output_df = final_pairs[[col for col in output_columns if col in final_pairs.columns]]

    # Convert types and format lists
    final_output_df = final_output_df.astype({
        'current_study_id': int, 'prior_study_id': int,
        'current_anchor_view': str, 'prior_anchor_view': str,
        'current_indication': str, 'current_findings': str, 'current_impression': str,
        'prior_indication': str, 'prior_findings': str, 'prior_impression': str,
        # Ensure path columns are strings (they should be already)
        'current_report_path': str, 'prior_report_path': str,
        'current_anchor_image': str, 'prior_anchor_image': str,
    })
    final_output_df['current_study_date'] = final_output_df['current_study_date'].dt.strftime('%Y-%m-%d')
    final_output_df['prior_study_date'] = final_output_df['prior_study_date'].dt.strftime('%Y-%m-%d')
    list_delimiter = '|'
    for col in ['current_aux_images', 'current_aux_views', 'prior_aux_images', 'prior_aux_views']:
         if col in final_output_df.columns:
             final_output_df[col] = final_output_df[col].apply(
                 lambda x: list_delimiter.join(map(str, x)) if isinstance(x, list) and x else ''
             ).astype(str) # Ensure final column is string

    # --- 13. Save ---
    logging.info(f"Saving final detailed report data with relative paths to: {output_path}")
    try:
        final_output_df.to_csv(output_path, index=False)
        logging.info("CSV file saved successfully.")
        return True
    except Exception as e:
        logging.error(f"Error saving data to CSV: {e}")
        return False

# --- Execute ---
if __name__ == '__main__':
    # Validate paths
    if not os.path.isfile(METADATA_CSV_PATH): logging.error(f"Metadata file missing: {METADATA_CSV_PATH}"); exit(1)
    if not os.path.isfile(SPLIT_CSV_PATH): logging.error(f"Split file missing: {SPLIT_CSV_PATH}"); exit(1)
    if not os.path.isdir(BASE_MIMIC_DIR): logging.error(f"Base MIMIC directory not found: {BASE_MIMIC_DIR}"); exit(1)
    # Check if the 'files' subdirectory exists under base dir
    files_dir_path = os.path.join(BASE_MIMIC_DIR, 'files')
    if not os.path.isdir(files_dir_path): logging.error(f"'files' directory not found under BASE_MIMIC_DIR: {files_dir_path}"); exit(1)

    success = prepare_longitudinal_mimiccxr_dataset(
        METADATA_CSV_PATH, SPLIT_CSV_PATH, BASE_MIMIC_DIR, OUTPUT_CSV_PATH
    )

    if success:
        print(f"\nSuccessfully created detailed longitudinal CSV with relative paths and revised report cleaning: {OUTPUT_CSV_PATH}")
    else:
        print("\nFailed to create detailed longitudinal CSV.")
        exit(1)

/home/users/nshaik3/miniconda3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-04-26 20:39:35,117 - INFO - Starting detailed longitudinal data preparation (relative paths, revised cleaning)...
2025-04-26 20:39:35,118 - INFO - Loading metadata: /data/users4/nshaik3/Datasets/MIMIC-CXR/physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-metadata.csv
2025-04-26 20:39:36,139 - INFO - Loading splits: /data/users4/nshaik3/Datasets/MIMIC-CXR/physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-split.csv
2025-04-26 20:39:36,621 - INFO - Merging metadata and split info...
2025-04-26 20:39:37,135 - INFO - Merged images: 377110
2025-04-26 20:39:37,137 - INFO - Cleaning merged data...
2025-04-26 20:39:37,696 - INFO - Cleaned images: 377110
2025-04-26 20:39:37,698 - INFO - Constructing relative image and 


Successfully created detailed longitudinal CSV with relative paths and revised report cleaning: /data/users4/nshaik3/Datasets/MIMIC-CXR/physionet.org/files/mimic-cxr-jpg/2.1.0/longitudinal_mimic-cxr-v1.0.csv
